# DSC PJATK Catalogue Quality Challenge — model

Baseline organizatorów z dwiema zmianami, każda sprawdzona osobnym porównaniem kontrolowanym
(ten sam model, hiperparametry i foldy po sprzedawcach; zmienia się jedna grupa cech):

| | Zmiana | OOF ROC-AUC | Average Precision |
|---|---|---|---|
| baseline | — | 0.7082 | 0.5706 |
| exp1 | `main_items`: liczba sztuk z tekstu bez gratisów, z liczbami słowami | 0.7324 | 0.6237 |
| exp2 | `OffTypical`: odległość ceny i wagi za sztukę od mediany tego samego produktu (mediany liczone w `fit`, czyli tylko na foldzie treningowym) | 0.9419 | 0.8717 |

Pod modelem: analiza 5 błędów OOF (reguła decyzji, macierz pomyłek, wybór przypadków).
Eksploracja danych, z której wynikły obie zmiany, jest w notebooku EDA.

Uruchomienie: dołącz dane konkursu i uruchom wszystkie komórki (CPU, bez internetu, ok. 25 s).
Kod pisany z pomocą asystenta AI (Claude); szczegóły w README.

In [ ]:
"""Baseline organizatorów + dwie zmiany: odczyt liczby sztuk (main_items) i odległość ceny/wagi od typowej (OffTypical)."""
import argparse
import json
import hashlib
import platform
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42


def seller_folds(sellers, n_splits=5, seed=SEED):
    """Stable group assignment across sklearn versions; no target information used."""
    assignment = np.array([int(hashlib.sha256(f'{seed}:{s}'.encode('utf-8')).hexdigest()[:16], 16)
                           % n_splits for s in sellers.astype(str)])
    for fold in range(n_splits):
        fit, valid = np.flatnonzero(assignment != fold), np.flatnonzero(assignment == fold)
        if not len(fit) or not len(valid):
            raise ValueError('Empty seller fold; use more groups or fewer folds')
        yield fit, valid


WORDS = {'one': '1', 'two': '2', 'three': '3', 'four': '4', 'five': '5'}
QTY = (r'(?:pack of|contains|main items:|quantity:?)\s*(\d+)'
       r'|(\d+)\s*(?:-pack|units|pcs|main items|individually)')


def main_items(text):
    """Liczba głównych sztuk z tekstu: pomija gratisy, rozumie liczby słowami."""
    text = text.lower()
    text = re.sub(r'also includes \w+ pcs of free accessory pieces', ' ', text)
    for word, digit in WORDS.items():
        text = re.sub(r'\b' + word + r'\b', digit, text)
    found = re.search(QTY, text)
    if found is None:
        return np.nan
    return float(found.group(1) or found.group(2))


def features(df):
    x = df.copy()
    title = x.title.fillna('').astype(str).str.lower()
    desc = x.description.fillna('').astype(str).str.lower()
    x['text'] = title + ' ' + desc
    x['title_length'] = title.str.len()
    x['description_length'] = desc.str.len()
    x['product'] = desc.str.extract(r'(?:^|\.\s*)([a-z][a-z \-]+?) by \w+', expand=False)
    # Missing observations remain missing; imputation is learned within each fold.
    for col in ['price_pln', 'shipping_weight_kg', 'pack_size']:
        x['log_' + col] = np.log1p(x[col].clip(lower=0))
    x['price_per_item'] = np.log1p(x.price_pln / x.pack_size.clip(lower=1))
    x['mass_per_item'] = np.log1p(x.shipping_weight_kg / x.pack_size.clip(lower=1))
    x['text_pack'] = x.text.apply(main_items)
    x['pack_difference'] = (x.text_pack - x.pack_size).abs()
    # Useful text-category interactions; no vocabulary or statistics learned here.
    x['category_text'] = [
        ' '.join(f'{cat}__{token}' for token in re.findall(r'\w+', text))
        for cat, text in zip(x.declared_category.fillna('missing'), x.text)
    ]
    return x.replace({pd.NA: np.nan})


class OffTypical(BaseEstimator, TransformerMixin):
    """How far price and weight per item are from what is typical for this product.
    Typical values (medians) are learned in fit(), i.e. only on the training fold."""

    def per_item(self, X):
        pack = X.pack_size.clip(lower=1)
        return pd.DataFrame({'product': X['product'],
                             'price': np.log(X.price_pln / pack),
                             'mass': np.log(X.shipping_weight_kg / pack)})

    def fit(self, X, y=None):
        self.typical_ = self.per_item(X).groupby('product').median()
        return self

    def transform(self, X):
        items = self.per_item(X)
        typical = self.typical_.reindex(items['product']).to_numpy()
        return np.abs(items[['price', 'mass']].to_numpy() - typical)


def make_model():
    numeric = ['log_price_pln', 'log_shipping_weight_kg', 'log_pack_size',
               'price_per_item', 'mass_per_item', 'text_pack', 'pack_difference',
               'title_length', 'description_length']
    categorical = ['declared_category', 'brand', 'condition']
    preprocessing = ColumnTransformer([
        ('text', TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=18000,
                                 sublinear_tf=True), 'text'),
        ('interactions', TfidfVectorizer(min_df=3, max_features=12000), 'category_text'),
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)),
                              ('scale', StandardScaler(with_mean=False))]), numeric),
        ('typical', Pipeline([('off', OffTypical()),
                              ('impute', SimpleImputer(strategy='median', add_indicator=True)),
                              ('scale', StandardScaler(with_mean=False))]),
         ['product', 'price_pln', 'shipping_weight_kg', 'pack_size']),
        ('categories', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                                 ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical)
    ], sparse_threshold=1.)
    return Pipeline([('preprocess', preprocessing),
                     ('model', LogisticRegression(C=2., max_iter=600, solver='liblinear',
                                                  random_state=SEED))])


def resolve_data_dir():
    local = Path('data/public')
    if (local / 'train.csv').exists():
        return local
    candidates = sorted(Path('/kaggle/input').rglob('train.csv'))
    candidates = [p.parent for p in candidates if (p.parent / 'sample_submission.csv').exists()]
    if len(candidates) != 1:
        raise ValueError('Specify --data-dir: could not identify one competition dataset')
    return candidates[0]


def run(data_dir, output_dir='baseline_output'):
    started = time.perf_counter()
    data_dir, output_dir = Path(data_dir), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(data_dir / 'train.csv')
    test = pd.read_csv(data_dir / 'test.csv')
    sample = pd.read_csv(data_dir / 'sample_submission.csv')
    assert train.target.isin([0, 1]).all()
    assert set(train.seller_id).isdisjoint(test.seller_id)
    assert train.id.is_unique and test.id.is_unique
    assert set(train.id).isdisjoint(test.id)
    assert sample.id.tolist() == test.id.tolist()
    print('Shapes:', train.shape, test.shape)
    print('Target proportions:', train.target.value_counts(normalize=True).to_dict())
    print('Missing fraction:', train.isna().mean().round(3).to_dict())
    print('Categories:', train.declared_category.value_counts(dropna=False).to_dict())
    print('Numeric distribution:\n', train.select_dtypes('number').describe().round(2))
    x, xt, y = features(train.drop(columns='target')), features(test), train.target.to_numpy()
    oof, predictions, scores = np.zeros(len(train)), np.zeros(len(test)), []
    fold_assignment = np.zeros(len(train), dtype=int)
    for fold, (fit, valid) in enumerate(seller_folds(train.seller_id), 1):
        assert set(train.iloc[fit].seller_id).isdisjoint(train.iloc[valid].seller_id)
        fold_assignment[valid] = fold
        model = make_model()
        model.fit(x.iloc[fit], y[fit])
        oof[valid] = model.predict_proba(x.iloc[valid])[:, 1]
        predictions += model.predict_proba(xt)[:, 1] / 5
        score = roc_auc_score(y[valid], oof[valid])
        scores.append(float(score))
        print(f'Fold {fold}: ROC-AUC={score:.6f}')
    report = {'fold_roc_auc': scores, 'mean_roc_auc': float(np.mean(scores)),
              'std_roc_auc': float(np.std(scores)), 'oof_roc_auc': float(roc_auc_score(y, oof)),
              'oof_average_precision': float(average_precision_score(y, oof)),
              'elapsed_seconds': time.perf_counter() - started,
              'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                           'pandas': pd.__version__, 'sklearn': sklearn.__version__},
              'seed': SEED,
              'fold_policy': 'SHA256(seed:seller_id) first 16 hex digits modulo 5',
              'data_sha256': {name: hashlib.sha256((data_dir / name).read_bytes()).hexdigest()
                              for name in ['train.csv', 'test.csv', 'sample_submission.csv']},
              'fold_assignment_sha256': hashlib.sha256(
                  pd.DataFrame({'id': train.id, 'fold': fold_assignment}).to_csv(index=False, lineterminator='\n').encode('utf-8')
              ).hexdigest()}
    sample['target'] = predictions
    assert np.isfinite(predictions).all() and ((predictions >= 0) & (predictions <= 1)).all()
    sample.to_csv(output_dir / 'submission.csv', index=False)
    pd.DataFrame({'id': train.id, 'target': y, 'prediction': oof,
                  'fold': fold_assignment}).to_csv(output_dir / 'oof.csv', index=False)
    (output_dir / 'cv_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))
    return report

## Validation and submission
Each fold fits its own imputation, TF-IDF vocabulary and model.
The CV score uses only listings held out from the corresponding model.
The submission averages five models; preserve this procedure when reproducing your result.

In [ ]:
data_dir = resolve_data_dir()
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('baseline_output')
report = run(data_dir, output_dir)
print('Submission:', output_dir / 'submission.csv')

## Analiza błędów OOF

**Reguła decyzji (budżet przeglądu):** moderator sprawdza tyle ofert, ile jest złych w train (17%, 2721 ofert),
zaczynając od najwyższego prawdopodobieństwa. Próg wynika tylko z predykcji OOF na train.

**Zasada wyboru przypadków:** 3 fałszywe alarmy (FP) z najwyższym p i 2 przegapione złe oferty (FN) z najniższym p,
czyli pomyłki, przy których model był najbardziej pewny. Bez losowania, więc wybór jest odtwarzalny.

In [ ]:
oof = pd.read_csv(output_dir / 'oof.csv')
budget = int(oof.target.sum())
threshold = oof.prediction.sort_values(ascending=False).iloc[budget - 1]
oof['decision'] = (oof.prediction >= threshold).astype(int)
print('Budżet przeglądu:', budget, 'ofert, próg:', round(threshold, 4))
print(pd.crosstab(oof.target, oof.decision, rownames=['prawda'], colnames=['decyzja']))

In [ ]:
train = pd.read_csv(data_dir / 'train.csv')
cases = oof.merge(train.drop(columns='target'), on='id')
fp = cases[(cases.decision == 1) & (cases.target == 0)].sort_values('prediction', ascending=False).head(3)
fn = cases[(cases.decision == 0) & (cases.target == 1)].sort_values('prediction').head(2)
five = pd.concat([fp, fn])

for r in five.itertuples():
    print(r.id, '| fold', r.fold, '| prawda', r.target, '| p =', round(r.prediction, 3), '| decyzja', r.decision)
    print('  kategoria:', r.declared_category, '| stan:', r.condition, '| cena:', r.price_pln, 'zł | waga:', r.shipping_weight_kg, 'kg | pack_size:', r.pack_size)
    print('  tytuł:', r.title)
    print('  opis:', r.description)
    print()

Dla każdego przypadku: typowa cena i waga za sztukę **tylko z innych foldów** (to, co model widział, zanim ocenił ofertę),
dla tego samego produktu, potem dodatkowo dla tego samego wariantu i stanu.

In [ ]:
x = features(train)
x['variant'] = x.description.str.lower().str.extract(r'variant: (\w+)', expand=False)
x['price_item'] = x.price_pln / x.pack_size.clip(lower=1)
x['kg_item'] = x.shipping_weight_kg / x.pack_size.clip(lower=1)

for r in five.itertuples():
    row = x[x.id == r.id].iloc[0]
    learn = x[(oof.fold != r.fold).to_numpy()]
    same = learn[learn['product'] == row['product']]
    same_var = same[same.variant == row.variant]
    same_var_cond = same_var[same_var.condition == row.condition]
    print(r.id, '|', row['product'], '| wariant:', row.variant, '| stan:', row.condition)
    print('  ta oferta:              ', round(row.price_item, 2), 'zł', round(row.kg_item, 3), 'kg')
    print('  typowy produkt:         ', round(same.price_item.median(), 2), 'zł', round(same.kg_item.median(), 3), 'kg   ofert:', len(same))
    print('  ten sam wariant:        ', round(same_var.price_item.median(), 2), 'zł', round(same_var.kg_item.median(), 3), 'kg   ofert:', len(same_var))
    print('  ten sam wariant i stan: ', round(same_var_cond.price_item.median(), 2), 'zł', round(same_var_cond.kg_item.median(), 3), 'kg   ofert:', len(same_var_cond))
    print()

**Wniosek (przypuszczenie, nie pewnik):** wszystkie 3 FP to wariant `large`, oba FN to `compact` i `used`.
Typowa wartość z `OffTypical` miesza warianty i stany: przy porównaniu z właściwą grupą różnica FP maleje, a FN rośnie.
Propozycja na kolejne porównanie: mediana dla produktu i wariantu. Ukrytego typu błędu nie znamy.